# 05 - Insights Summary

## 项目背景

本notebook汇总01-04的全部分析发现，将数据洞察转化为可执行的产品优化建议。面向的读者是产品负责人和技术团队，目标是回答三个问题：
1. 产品现在最大的问题是什么
2. 应该优先优化哪里
3. 怎么持续监控优化效果

## 内容结构

- 核心指标全景
- 各场景问题诊断与优先级排序
- 可执行的优化建议
- 监控指标体系建议

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'

INTENT_CN = {
    'code_generation': '代码生成', 'knowledge_qa': '知识问答',
    'creative_writing': '创意写作', 'translation': '翻译',
    'chitchat': '闲聊', 'data_analysis': '数据分析'
}

# 加载数据
convs = pd.read_csv('../data/conversations.csv', parse_dates=['timestamp'])
events = pd.read_csv('../data/events.csv', parse_dates=['event_timestamp'])
users = pd.read_csv('../data/users.csv', parse_dates=['signup_date'])

print('数据加载完成')

In [ ]:
# === 核心指标全景 ===

# 计算核心指标
total_convs = len(convs)
total_users = len(users)

conv_with_events = events['conversation_id'].unique()
silent_count = len(convs[~convs['conversation_id'].isin(conv_with_events)])
silent_rate = silent_count / total_convs * 100

first_events = events.sort_values('event_timestamp').groupby('conversation_id').first().reset_index()
like_rate = (first_events['event_type'] == 'like').sum() / total_convs * 100
dislike_rate = (first_events['event_type'] == 'dislike').sum() / total_convs * 100
retry_rate = (first_events['event_type'] == 'retry').sum() / total_convs * 100
followup_rate = (first_events['event_type'] == 'followup').sum() / total_convs * 100

metrics = [
    ('总用户数', f'{total_users}', '#6366f1'),
    ('总对话数', f'{total_convs:,}', '#6366f1'),
    ('点赞率', f'{like_rate:.1f}%', '#2ecc71'),
    ('点踩率', f'{dislike_rate:.1f}%', '#e74c3c'),
    ('重新生成率', f'{retry_rate:.1f}%', '#f39c12'),
    ('追问率', f'{followup_rate:.1f}%', '#3498db'),
    ('沉默率', f'{silent_rate:.1f}%', '#95a5a6'),
]

from IPython.display import HTML

cards = '<div style="display: flex; flex-wrap: wrap; gap: 12px; font-family: -apple-system, sans-serif;">'
for label, value, color in metrics:
    cards += f'''
    <div style="flex: 1; min-width: 120px; background: #f9f9f9; border-top: 3px solid {color};
                padding: 16px; border-radius: 0 0 8px 8px; text-align: center;">
        <div style="font-size: 24px; font-weight: bold; color: {color};">{value}</div>
        <div style="font-size: 13px; color: #666; margin-top: 4px;">{label}</div>
    </div>
    '''
cards += '</div>'

HTML(cards)

In [ ]:
# === 各场景问题诊断与优先级 ===

# 计算各场景核心指标
first_with_intent = first_events.merge(convs[['conversation_id', 'intent']], on='conversation_id')
silent_convs = convs[~convs['conversation_id'].isin(conv_with_events)][['conversation_id', 'intent']].copy()
silent_convs['event_type'] = 'silent'
combined = pd.concat([first_with_intent[['conversation_id', 'intent', 'event_type']], silent_convs])

# 按场景统计
scene_stats = []
for intent, intent_cn in INTENT_CN.items():
    subset = combined[combined['intent'] == intent]
    total = len(subset)
    dislike = (subset['event_type'] == 'dislike').sum() / total * 100
    retry = (subset['event_type'] == 'retry').sum() / total * 100
    like = (subset['event_type'] == 'like').sum() / total * 100
    dissatisfaction = dislike + retry
    
    # 纠错式追问占比（从04的逻辑）
    fu = events[(events['event_type'] == 'followup') & (events['followup_text'] != '')]
    fu = fu.merge(convs[['conversation_id', 'intent']], on='conversation_id')
    fu_intent = fu[fu['intent'] == intent]
    corrective_keywords = ['不是', '你理解错', '有问题', '不对', '你没有回答', '矛盾', '不是这个意思', '明显不对', '能不能直接', '太笼统']
    if len(fu_intent) > 0:
        corrective_rate = fu_intent['followup_text'].apply(lambda x: any(kw in x for kw in corrective_keywords)).mean() * 100
    else:
        corrective_rate = 0
    
    scene_stats.append({
        '场景': intent_cn,
        '对话占比': f'{total/len(combined)*100:.0f}%',
        '点赞率': f'{like:.1f}%',
        '不满意率': f'{dissatisfaction:.1f}%',
        '纠错追问率': f'{corrective_rate:.0f}%',
        '_sort': dissatisfaction
    })

# 定义优先级排序
priority_order = ['代码生成', '知识问答', '数据分析', '创意写作', '翻译', '闲聊']
scene_df = pd.DataFrame(scene_stats).drop('_sort', axis=1)
scene_df['_order'] = scene_df['场景'].map({name: i for i, name in enumerate(priority_order)})
scene_df = scene_df.sort_values('_order').drop('_order', axis=1)

# 诊断卡片
diagnosis = {
    '代码生成': ('retry率最高，用户反复重新生成', '幻觉 + 拒绝回答', '🔴 最高'),
    '知识问答': ('dislike率最高，纠错追问64%', '幻觉（22%）为核心问题', '🔴 最高'),
    '数据分析': ('格式不匹配严重（65%）', '模型给建议而非直接给方案', '🟡 中等'),
    '创意写作': ('拒绝回答占23%', '过度安全策略', '🟡 中等'),
    '翻译': ('整体表现稳定', '无突出问题', '🟢 低'),
    '闲聊': ('沉默率65%但符合预期', '场景特性，非产品问题', '🟢 低'),
}

from IPython.display import HTML

html = '<div style="font-family: -apple-system, sans-serif;">'
html += '<h3>各场景问题诊断与优先级</h3>'

for _, row in scene_df.iterrows():
    scene = row['场景']
    symptom, root_cause, priority = diagnosis[scene]
    
    if '最高' in priority:
        border_color = '#e74c3c'
        bg = '#fef2f2'
    elif '中等' in priority:
        border_color = '#f39c12'
        bg = '#fffbeb'
    else:
        border_color = '#2ecc71'
        bg = '#f0fdf4'
    
    html += f'''
    <div style="border-left: 4px solid {border_color}; background: {bg}; padding: 14px 18px;
                margin-bottom: 12px; border-radius: 0 8px 8px 0;">
        <div style="display: flex; justify-content: space-between; align-items: center;">
            <span style="font-size: 16px; font-weight: bold;">{scene}</span>
            <span style="font-size: 13px;">优先级: {priority}</span>
        </div>
        <div style="font-size: 13px; color: #555; margin-top: 8px;">
            <b>表现:</b> 不满意率 {row['不满意率']} | 点赞率 {row['点赞率']} | 纠错追问率 {row['纠错追问率']}
        </div>
        <div style="font-size: 13px; color: #555; margin-top: 4px;">
            <b>症状:</b> {symptom}
        </div>
        <div style="font-size: 13px; color: #555; margin-top: 4px;">
            <b>根因:</b> {root_cause}
        </div>
    </div>
    '''

html += '</div>'
HTML(html)

## 优化建议

### 🔴 P0 — 立即优化

**知识问答：解决幻觉问题**
- 接入RAG（检索增强生成），让模型基于可靠知识源回答
- 对涉及事实、数据、人物的回答增加置信度检测，低置信度时主动标注"我不确定"
- 预期收益：知识问答dislike率从24%降低，纠错式追问率从64%下降

**代码生成：减少拒答和无效回复**
- 接入代码执行沙箱，让模型生成后自动验证代码可运行
- 优化prompt策略：用户要代码时直接给代码，不要只给思路
- 预期收益：代码生成retry率从30%降低

### 🟡 P1 — 短期优化

**数据分析：改善格式匹配**
- 检测到数据分析类query时，默认输出包含代码块的结构化回答
- 减少"请提供数据"类回复，改为先给通用方案再询问是否需要定制
- 预期收益：格式不匹配率从65%降低

**创意写作：放宽安全策略**
- 审查当前安全规则中对创意写作的限制条件，减少误拒
- 风格模仿类请求（如"用鲁迅风格写"）不应触发版权拒绝
- 预期收益：拒绝回答率从23%降低

### 🟢 P2 — 持续关注

**翻译：保持现状，监控波动**

**闲聊：沉默率高符合场景特性，无需专项优化**

## 监控指标体系建议

| 层级 | 指标 | 监控频率 | 告警阈值 |
|------|------|---------|---------|
| 整体健康度 | DAU、留存率、人均对话数 | 每日 | DAU日环比下降>10% |
| 回复质量 | 点赞率、点踩率、retry率 | 每日 | 点踩率日环比上升>5% |
| 场景质量 | 各场景不满意率 | 每周 | 任一场景不满意率>50% |
| 问题归因 | 各归因类别占比 | 每周 | 幻觉占比>15% |
| 追问信号 | 纠错式追问率 | 每周 | 纠错率>50% |

## 项目总结

### 分析框架回顾

本项目构建了一套完整的对话式AI产品用户行为分析流水线：

| Notebook | 解决的问题 | 核心方法 |
|----------|-----------|---------|
| 01 数据生成 | 没有真实数据怎么办 | 基于场景差异化配置的合成数据，事件流模型 |
| 02 行为分析 | 产品整体表现如何 | 行为分布、场景交叉分析、留存曲线、时间趋势 |
| 03 语义归因 | 为什么用户不满意 | 关键词规则 + LLM双路径归因，bad case根因分类 |
| 04 追问拆解 | 追问是好事还是坏事 | 探索式/纠错式二分法，追问与场景质量的关联 |
| 05 洞察总结 | 应该优先做什么 | 场景诊断、优先级排序、可执行建议、监控体系 |

### 关键结论

1. **代码生成和知识问答是当前最需要优化的两个场景**，合计贡献了超过60%的bad case
2. **不同场景的问题根因完全不同**——知识问答是幻觉，代码生成是拒答，数据分析是格式不匹配——不能用同一个方案解决
3. **追问率需要拆分监控**——整体追问率19.4%看似正常，但知识问答场景64%的追问是纠错式的，这是一个隐藏的质量危机
4. **沉默是最大的盲区**——24%的对话没有任何反馈，这部分用户的满意度完全未知

### 局限性

- 合成数据的行为模式由预设概率驱动，真实用户行为更复杂
- 关键词规则归因覆盖率有限（71%兜底到质量不足），生产环境应使用LLM打标
- 留存曲线末端样本量不足导致Day 30数据偏差
- 追问后二次行为的差异不显著，源于合成数据事件流的关联逻辑过于简单

### 可扩展方向

- 接入真实产品数据替换合成数据，验证分析框架的有效性
- 引入A/B测试模块，量化优化措施的实际效果
- 构建实时监控看板（如Grafana），将本项目的指标体系产品化
- 加入用户分群维度（新用户 vs 老用户），细化不同人群的质量感知差异